<a href="https://colab.research.google.com/github/Sav200512/sp500-rfm-segmentation-cw/blob/main/Task_1_and_2_cw_(20232749_W2151870).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Task 1.1 Ticker selection and data acquisition

In [21]:
import os, json, platform
from io import StringIO
import requests
import numpy as np
import pandas as pd
import yfinance as yf

In [22]:
SEED = 42
START, END = "2021-01-01", "2025-01-01"   # END is exclusive
BENCH = "^GSPC"

for folder in ["data/raw", "data/processed", "figures"]:
    os.makedirs(folder, exist_ok=True)

print("pandas", pd.__version__, "| numpy", np.__version__, "| yfinance", yf.__version__)

pandas 2.2.3 | numpy 2.4.6 | yfinance 0.2.66


In [23]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
sp500 = pd.read_html(StringIO(html))[0]

sp500["Yahoo_Symbol"] = sp500["Symbol"].str.replace(".", "-", regex=False)
sp500["Retrieved_On"] = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC")

sp500_out = sp500[["Symbol", "Yahoo_Symbol", "Security", "GICS Sector", "Retrieved_On"]]
sp500_out.to_csv("data/raw/sp500_tickers.csv", index=False)

print("Tickers:", len(sp500_out), "| retrieved:", sp500_out["Retrieved_On"].iloc[0])
sp500_out.head()

Tickers: 503 | retrieved: 2026-11-11 04:41 UTC


,Symbol,Yahoo_Symbol,Security,GICS Sector,Retrieved_On
0,MMM,MMM,3M,Industrials,2026-11-11 04:41 UTC
1,AOS,AOS,A. O. Smith,Industrials,2026-11-11 04:41 UTC
2,ABT,ABT,Abbott Laboratories,Health Care,2026-11-11 04:41 UTC
3,ABBV,ABBV,AbbVie,Health Care,2026-11-11 04:41 UTC
4,ACN,ACN,Accenture,Information Technology,2026-11-11 04:41 UTC


In [24]:
tickers = sp500["Yahoo_Symbol"].tolist()
frames = []

for i in range(0, len(tickers), 50):
    batch = tickers[i:i + 50]
    d = yf.download(batch, start=START, end=END, auto_adjust=False,
                    progress=False, threads=True)
    frames.append(d)
    print(f"Batch {i // 50 + 1} done ({len(batch)} tickers)")

raw = pd.concat(frames, axis=1)

close = raw["Close"]
adj_close = raw["Adj Close"]
volume = raw["Volume"]

# Benchmark
bench = yf.download(BENCH, start=START, end=END, auto_adjust=False, progress=False)
if isinstance(bench.columns, pd.MultiIndex):
    bench.columns = bench.columns.get_level_values(0)

Batch 1 done (50 tickers)
Batch 2 done (50 tickers)
Batch 3 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 4 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 5 done (50 tickers)
Batch 6 done (50 tickers)
Batch 7 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 8 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 9 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['VYLR']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 10 done (50 tickers)
Batch 11 done (3 tickers)


In [25]:
close.to_csv("data/raw/close_daily.csv.gz")
adj_close.to_csv("data/raw/adj_close_daily.csv.gz")
volume.to_csv("data/raw/volume_daily.csv.gz")
bench.to_csv("data/raw/gspc_daily.csv")

run_log = {
    "download_time_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "start": START, "end_exclusive": END, "benchmark": BENCH,
    "yfinance": yf.__version__, "pandas": pd.__version__,
    "python": platform.python_version(),
    "tickers_requested": len(tickers),
}
json.dump(run_log, open("data/raw/run_log.json", "w"), indent=2)

In [26]:
no_data = [t for t in tickers if t not in adj_close.columns or adj_close[t].isna().all()]

print("Requested:", len(tickers))
print("Returned with some data:", len(tickers) - len(no_data))
print("No data:", no_data)
print("Shares: first date", adj_close.index.min().date(), "| last date", adj_close.index.max().date())
print("Benchmark: first date", bench.index.min().date(), "| last date", bench.index.max().date(), "| rows", len(bench))
print("Share rows:", len(adj_close), "| shape close/adj/vol:", close.shape, adj_close.shape, volume.shape)

Requested: 503
Returned with some data: 498
No data: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
Shares: first date 2021-01-04 | last date 2024-12-31
Benchmark: first date 2021-01-04 | last date 2024-12-31 | rows 1005
Share rows: 1005 | shape close/adj/vol: (1005, 503) (1005, 503) (1005, 503)


Task 1.2 Data validation and alignment

In [27]:
# Index-level checks
print("Duplicate dates (shares):", adj_close.index.duplicated().sum())
print("Duplicate dates (benchmark):", bench.index.duplicated().sum())
print("Dates sorted:", adj_close.index.is_monotonic_increasing)
print("Share and benchmark dates identical:", adj_close.index.equals(bench.index))

# Per-share diagnostics
cols = adj_close.columns
diag = pd.DataFrame(index=cols)
diag["first_valid"]     = adj_close.apply(lambda s: s.first_valid_index())
diag["last_valid"]      = adj_close.apply(lambda s: s.last_valid_index())
diag["n_obs"]           = adj_close.notna().sum()
diag["miss_adj"]        = adj_close.isna().sum()
diag["miss_close"]      = close[cols].isna().sum()
diag["miss_vol"]        = volume[cols].isna().sum()
diag["nonpos_price"]    = ((adj_close <= 0) | (close[cols] <= 0)).sum()
diag["neg_vol"]         = (volume[cols] < 0).sum()
diag["zero_vol"]        = (volume[cols] == 0).sum()

print("Trading sessions:", len(adj_close))
print(diag.describe().T[["min", "max"]])
diag[diag["n_obs"] < len(adj_close)].sort_values("n_obs")

Duplicate dates (shares): 0
Duplicate dates (benchmark): 0
Dates sorted: True
Share and benchmark dates identical: True
Trading sessions: 1005
                              min                  max
first_valid   2021-01-04 00:00:00  2024-03-27 00:00:00
last_valid    2024-12-31 00:00:00  2024-12-31 00:00:00
n_obs                         0.0               1005.0
miss_adj                      0.0               1005.0
miss_close                    0.0               1005.0
miss_vol                      0.0               1005.0
nonpos_price                  0.0                  0.0
neg_vol                       0.0                  0.0
zero_vol                      0.0                424.0


,first_valid,last_valid,n_obs,miss_adj,miss_close,miss_vol,nonpos_price,neg_vol,zero_vol
Ticker,,,,,,,,,
HONA,NaT,NaT,0,1005,1005,1005,0,0,0
FDXF,NaT,NaT,0,1005,1005,1005,0,0,0
SNDK,NaT,NaT,0,1005,1005,1005,0,0,0
Q,NaT,NaT,0,1005,1005,1005,0,0,0
VYLR,NaT,NaT,0,1005,1005,1005,0,0,0
GEV,2024-03-27,2024-12-31,193,812,812,812,0,0,0
SOLV,2024-03-26,2024-12-31,194,811,811,811,0,0,0
RDDT,2024-03-21,2024-12-31,197,808,808,808,0,0,0
VLTO,2023-10-04,2024-12-31,313,692,692,692,0,0,0


In [28]:
# Large one-day moves in Close can signal an unadjusted split or a data error.
close_ret = close.pct_change(fill_method=None)
big = (close_ret.abs() > 0.40).sum()
big = big[big > 0].sort_values(ascending=False)
print("Shares with a one-day Close move > 40%:", len(big))
print(big)

# Compare Adj Close / Close: it should change only on dividend dates.
ratio = adj_close / close[cols]
print("Shares where Adj Close differs from Close on at least one date:", (ratio.sub(1).abs().max() > 1e-9).sum())

# Spot-check a known split (example: NVDA 10-for-1, June 2024)
print(close["NVDA"].loc["2024-06-05":"2024-06-14"])
print(adj_close["NVDA"].loc["2024-06-05":"2024-06-14"])
print(volume["NVDA"].loc["2024-06-05":"2024-06-14"])

Shares with a one-day Close move > 40%: 7
Ticker
CVNA    5
APP     1
BE      1
DXCM    1
GL      1
HOOD    1
RDDT    1
dtype: int64
Shares where Adj Close differs from Close on at least one date: 405
Date
2024-06-05    122.440002
2024-06-06    120.998001
2024-06-07    120.888000
2024-06-10    121.790001
2024-06-11    120.910004
2024-06-12    125.199997
2024-06-13    129.610001
2024-06-14    131.880005
Name: NVDA, dtype: float64
Date
2024-06-05    122.091858
2024-06-06    120.653969
2024-06-07    120.544273
2024-06-10    121.443710
2024-06-11    120.576134
2024-06-12    124.854271
2024-06-13    129.252090
2024-06-14    131.515839
Name: NVDA, dtype: float64
Date
2024-06-05    528402000
2024-06-06    664696000
2024-06-07    412386000
2024-06-10    313434100
2024-06-11    222551200
2024-06-12    299595000
2024-06-13    260704500
2024-06-14    309320400
Name: NVDA, dtype: int64


In [29]:
n_sessions = len(adj_close)
reasons = {}

for t, r in diag.iterrows():
    why = []
    if r["n_obs"] == 0:
        why.append("no price data in study period")
    elif r["n_obs"] < n_sessions:
        why.append(f"unequal history ({int(r['n_obs'])}/{n_sessions} sessions)")
    if r["miss_close"] > 0 or r["miss_vol"] > 0:
        why.append("missing Close or Volume")
    if r["nonpos_price"] > 0:
        why.append("nonpositive price")
    if r["neg_vol"] > 0:
        why.append("negative volume")
    if why:
        reasons[t] = "; ".join(why)

exclusion_log = pd.DataFrame({"Ticker": list(reasons), "Reason": list(reasons.values())})
included = [t for t in cols if t not in reasons]

exclusion_log.to_csv("data/processed/exclusion_log.csv", index=False)
pd.Series(included, name="Ticker").to_csv("data/processed/included_shares.csv", index=False)

print("Included:", len(included), "| Excluded:", len(exclusion_log))
print(exclusion_log["Reason"].value_counts())
print("Shares with any zero-volume days (not excluded, check):", int((diag.loc[included, "zero_vol"] > 0).sum()))

Included: 488 | Excluded: 15
Reason
no price data in study period; missing Close or Volume          5
unequal history (742/1005 sessions); missing Close or Volume    1
unequal history (935/1005 sessions); missing Close or Volume    1
unequal history (936/1005 sessions); missing Close or Volume    1
unequal history (979/1005 sessions); missing Close or Volume    1
unequal history (513/1005 sessions); missing Close or Volume    1
unequal history (193/1005 sessions); missing Close or Volume    1
unequal history (862/1005 sessions); missing Close or Volume    1
unequal history (197/1005 sessions); missing Close or Volume    1
unequal history (194/1005 sessions); missing Close or Volume    1
unequal history (313/1005 sessions); missing Close or Volume    1
Name: count, dtype: int64
Shares with any zero-volume days (not excluded, check): 2


In [30]:
adj_inc = adj_close[included]
bench_adj = bench["Adj Close"]

w_shares = adj_inc.resample("W-FRI").last()
w_bench  = bench_adj.resample("W-FRI").last()

# Drop incomplete first/last calendar weeks (Mon-Fri window must sit inside the study period)
last_date = adj_close.index.max()
week_start = w_shares.index - pd.Timedelta(days=4)
keep = (week_start >= pd.Timestamp(START)) & (w_shares.index <= last_date)
w_shares, w_bench = w_shares[keep], w_bench[keep]

assert w_shares.index.equals(w_bench.index), "Week labels differ"

ret_shares = w_shares.pct_change(fill_method=None).iloc[1:]
ret_bench  = w_bench.pct_change(fill_method=None).iloc[1:]

print("Weekly price labels:", w_shares.index.min().date(), "to", w_shares.index.max().date(), f"({len(w_shares)})")
print("Weekly return observations:", len(ret_shares))
print("Any NaN in share returns:", ret_shares.isna().any().any())

Weekly price labels: 2021-01-08 to 2024-12-27 (208)
Weekly return observations: 207
Any NaN in share returns: False
